# **Ajuste Fino Eficiente em Modelos de Linguagem**
<font size=3>
    
Na aula anterior, desvendamos a arquitetura interna do modelo e realizamos o seu pré-treinamento *do zero* utilizando a técnica de *Masked Language Modeling* (MLM). Durante aquele processo computacionalmente intensivo, o nosso modelo construiu uma base fundacional sólida, aprendendo as regras sintáticas e as complexas representações semânticas bidirecionais da linguagem.

O nosso objetivo nesta aula é transferir esse conhecimento prévio para resolver um problema específico do mundo real (*downstream task*): a **análise de sentimentos** em críticas de cinema do dataset IMDB. Contudo, em vez de seguirmos o caminho clássico e dispendioso de reajustar todos os pesos da rede neural, iremos estudar e implementar o paradigma **PEFT** (*Parameter-Efficient Fine-Tuning*) através do algoritmo **LoRA** (*Low-Rank Adaptation*), ensinando o nosso modelo a dominar uma nova tarefa de forma matematicamente elegante, extremamente rápida e com um custo de memória drasticamente reduzido.

## **1. _Fine-Tuning_ Tradicional _vs_ o Paradigma PEFT:**
<font size=3>
    
Até o surgimento dos Grandes Modelos de Linguagem (LLMs), a abordagem padrão para transferir conhecimento de um modelo pré-treinado consistia no **_Full Fine-Tuning_** (Ajuste Fino Completo). Nesse cenário, inicializamos o modelo com os pesos do pré-treinamento e, durante o treinamento na nova tarefa, **todas as camadas e parâmetros originais são atualizados** por meio do algoritmo de retropropagação (*backpropagation*).

<font size=3>

Embora produza excelentes resultados em métricas de acurácia, o *Full Fine-Tuning* apresenta alguns problemas de engenharia quando a escala do modelo cresce:

1. **Consumo extremo de memória de GPU:** Durante o treinamento, a GPU não armazena apenas os pesos do modelo. Para calcular a atualização, precisamos manter na memória de vídeo (VRAM) as **ativações intermediárias** (para o *backward pass*), os **gradientes** de cada parâmetro e os **estados do otimizador**. O otimizador Adam, por exemplo, armazena momentos de primeira e segunda ordem para *cada parâmetro treinável*. Isso significa que treinar um modelo exige várias vezes mais memória do que apenas rodá-lo (inferência).

2. **Custo de armazenamento e implantação:** Se uma equipe quiser usar o mesmo modelo fundacional para 10 tarefas diferentes (análise de sentimentos, tradução, sumarização, etc.), o *full fine-tuning* exigirá salvar 10 cópias completas do modelo. Em modelos com bilhões de parâmetros, isso torna o armazenamento e a alocação em servidores de produção insustentáveis.

3. **Esquecimento catastrófico (*catastrophic forgetting*):** Ao forçar a rede a ajustar agressivamente todos os seus pesos para uma tarefa muito específica, ela corre o risco de degradar ou "esquecer" as representações genéricas robustas (sintaxe, gramática, fatos do mundo) que custaram caro para serem aprendidas durante o pré-treinamento.

### **1.1 O Paradigma PEFT:**
<font size=3>
    
Para resolver esse gargalo computacional, a comunidade científica de PLN desenvolveu o conceito de **Ajuste Fino Eficiente em Parâmetros** (*Parameter-Efficient Fine-Tuning* — PEFT). A abordagem do PEFT é sofisticada e simples:
>**Manter os pesos originais do modelo pré-treinado 100% congelados** e introduzir (ou selecionar) uma quantidade ínfima de parâmetros adicionais — geralmente entre $0.1\%$ e $1\%$ do total — que serão os únicos atualizados durante o treinamento.

As técnicas de PEFT conseguem alcançar desempenhos comparáveis ao *full fine-tuning*, mas com uma fração do custo computacional. Elas se dividem em três grandes famílias estratégicas:

* **Métodos aditivos:** Introduzem fisicamente pequenos blocos neurais auxiliares (*adapters*) entre as camadas originais do *Transformer*, ou adicionam tokens virtuais treináveis no início da sequência de entrada (*Prompt Tuning* e *Prefix Tuning*). O grande problema dessa abordagem é que ela introduz **latência na inferência** (maior tempo para gerar a predição), pois os dados precisam passar por mais camadas físicas durante a execução.

* **Métodos seletivos:** Não adicionam novos parâmetros. Em vez disso, usam critérios para escolher um subconjunto minúsculo dos parâmetros originais (como apenas as camadas de *bias* de toda a rede) para serem descongelados e treinados. Um exemplo clássico é o algoritmo *BitFit*.

* **Métodos de reparametrização:** Utilizam propriedades matemáticas e transformações de álgebra linear para mapear a atualização dos pesos em espaços de dimensões muito menores. É exatamente nesta categoria que reside o **LoRA**, a técnica mais elegante e utilizada pela indústria atualmente, que implementaremos a seguir.

## **2. _Low-Rank Adaptation_:**
<font size=3>
    
O **LoRA** (*Low-Rank Adaptation*), introduzido por [Hu et al. em 2021](https://arxiv.org/abs/2106.09685), fundamenta-se em um conceito de álgebra linear associado à hipótese da **Dimensão Intrínseca Baixa**:
> *Embora os modelos de linguagem possuam matrizes de pesos gigantescas (com bilhões de parâmetros), as alterações necessárias para adaptar esses pesos a uma tarefa específica vivem em um subespaço de dimensão muito menor.*

### **2.1. A formulação matemática:**
<font size=3>

Considere uma camada linear genérica do nosso `MiniBERT` (por exemplo, a projeção de *Query* ou as camadas do *Feed-Forward Network*). A operação original de propagação (*forward pass*) para uma entrada $x$ é dada por
$$
    h = W_0\, x \, ,
$$
onde:
* $W_0 \in \mathbb{R}^{d \times k}$ é a matriz de pesos pré-treinada original.
* $x \in \mathbb{R}^{k \times 1}$ é o vetor de entrada (representando um token).

No *full fine-tuning*, nós treinaríamos e atualizaríamos toda a matriz $W_0$. Aqui, a matriz atualizada passará a ser $W = W_0 + \Delta W$, onde a **matriz de atualização de pesos** $\Delta W$ tem exatamente a mesma dimensão da matriz original $\left(\Delta W \in \mathbb{R}^{d \times k}\right)$.

<font size=3>

Em vez de aprender a matriz densa $\Delta W$ completa, o **LoRA fatora a atualização no produto de duas matrizes:**
$$
    \Delta W = B \cdot A \,
$$

Nós definimos um hiperparâmetro $r$ (*rank*), escolhendo $r \ll \min(d,\, k)$. A partir disso, instanciamos duas novas matrizes treináveis:

* A matriz **$A \in \mathbb{R}^{r \times k}$** (responsável por "comprimir" a entrada).
* A matriz **$B \in \mathbb{R}^{d \times r}$** (responsável por "projetar" de volta para a dimensão de saída).

A nova transformação linear da nossa camada passa a ser computada como a soma de dois fluxos de informação paralelos — o **fluxo congelado** e o **fluxo adaptativo**,
$$
    h = W_0\, x + \Delta W\, x = W_0\, x + \frac{\alpha}{r} (B \cdot A)\, x \, ,
$$
onde:
* O fluxo original $W_0 \,x$ permanece 100% congelado durante o treino.

* O hiperparâmetro $\alpha$ é uma constante de escalonamento. O termo $\alpha/r$ garante que a magnitude da ativação do adaptador permaneça estável independentemente do valor de $r$ que escolhermos.

* **A redução de parâmetros é drástica:** Se $d = 512$ e $k = 512$, a matriz original $\Delta W$ exigiria o treinamento de $262.144$ parâmetros. Se usarmos um *rank* $r = 4$, as matrizes $B$ e $A$ juntas terão apenas $(512 \times 4) + (4 \times 512) = 4.096$ parâmetros. **Isso representa uma redução de 98,4% no custo de memória.**

### **2.2. O Segredo da Inicialização:**
<font size=3>

Para evitar que as novas matrizes causem instabilidade no início do treinamento, o LoRA define uma regra de inicialização matemática estrita:

1. A matriz $A$ é inicializada a partir de uma distribuição Gaussiana $\mathcal{N}(0,\, \sigma^2)$.
2. A matriz $B$ é inicializada **inteiramente com zeros**.

> Na primeira iteração do treinamento, o produto $\Delta W = B \cdot A$ resultará rigorosamente na matriz nula. Isso garante que $h = W_0\, x + 0$, ou seja, o modelo começa a tarefa nova se comportando *exatamente* como o modelo pré-treinado original.



### **2.3. Implementação numérica:**
<font size=3>

Vamos transcrever a equação $h = W_0\, x + \dfrac{\alpha}{r}(B\cdot A)\,x$ para uma camada customizada chamada `LoRADense`. Ela receberá uma camada `Dense` original, congelará seus pesos e adicionará o fluxo adaptativo.

In [ ]:
import tensorflow as tf
from keras import layers, initializers

In [ ]:
class LoRADense(layers.Layer):

    def __init__(self, original_layer, r=4, alpha=8):
        super().__init__()

        self.original_layer = original_layer # fluxo W_0
        self.original_layer.trainable = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha/r

        # propriedades extraídas da camada original para manter coerência:
        self.units = original_layer.units

    def build(self, input_shape):

        k = input_shape[-1] # dimensão de entrada

        # matriz A (k, r) - inicialização aleatória gaussiana:
        self.lora_A = self.add_weight(name='lora_A',
                                      shape=(k, self.r),
                                      initializer=initializers.RandomNormal(stddev=1.0/self.r),
                                      trainable=True)

        # matriz B (r, d) - inicialização com Zeros:
        self.lora_B = self.add_weight(name='lora_B',
                                      shape=(self.r, self.units),
                                      initializer='zeros',
                                      trainable=True)

        super().build(input_shape)


    def call(self, x):

        base_output = self.original_layer(x) # W_0·x

        # A ordem de multiplicação obedece às dimensões dos tensores do TF (batch, seq, dim),
        # x: (..., k); A: (k, r); B (r, d)
        lora_output = tf.matmul(x, self.lora_A)
        lora_output = tf.matmul(lora_output, self.lora_B)

        # h = W_0 x + (B·A)·x*(alpha/r)
        return base_output + (lora_output*self.scaling)


## **3. Injeção do LoRA no MiniBERT:**
<font size=3>
    
Com a nossa camada `LoRADense` totalmente funcional, o próximo passo é recuperar o modelo `MiniBERT` que pré-treinámos na aula anterior. O nosso objetivo nesta secção divide-se em três etapas claras:

1. **Congelamento global:** Imobilizar todos os pesos originais do modelo para garantir a eficiência do PEFT e evitar o *esquecimento catastrófico*.

2. **Injeção na Atenção:** Substituir especificamente as projeções de *Query* ($W^Q$) e *Value* ($W^V$) de cada bloco de atenção pelo nosso adaptador LoRA.

3. **Construção do _Head_ de Classificação:** Adicionar uma camada de saída ligada ao token `[CLS]` (`pooled_output`) para resolver a nossa tarefa de análise de sentimentos.
   

### **3.1. Reconstruindo e congelando o modelo base:**
<font size=3>

Primeiro, carregamos as configurações estruturais do nosso modelo a partir do arquivo JSON gerado no pré-treino e instanciamos a arquitetura base para carregar os pesos pré-treinados salvos.

In [ ]:
import json
from keras import Model
from mybert import MiniBERT

In [ ]:
# carregando as configurações estruturais do modelo:
with open('weights/minibert_config.json', 'r') as f:
    config = json.load(f)

config

In [ ]:
# instanciando o modelo MiniBERT original:
base_minibert = MiniBERT(**config)

# criando um mini-batch fictício de tamanho 1 para dar o "Build" no modelo:
dummy_inp = tf.zeros((1, config["max_len"]), dtype=tf.int32)
dummy_seg = tf.zeros((1, config["max_len"]), dtype=tf.int32)
dummy_mask = tf.ones((1, config["max_len"]), dtype=tf.int32)

# executando uma passada rápida (sem treinar) para que o Keras crie as matrizes de pesos:
_ = base_minibert((dummy_inp, dummy_seg, dummy_mask), training=False)

# carregando os pesos resultantes do pré-treinamento por MLM:
base_minibert.load_weights('weights/minibert.weights.h5')

# congelamento global:
base_minibert.trainable = False

### **3.2 A injeção do LoRA:**
<font size=3>

De acordo com o artigo original do LoRA (Hu et al., 2021), os maiores ganhos de qualidade de adaptação em modelos baseados em *Transformers* ocorrem quando aplicamos o algoritmo nas matrizes de projeção do mecanismo de atenção. Especificamente, focar em **Query ($W^Q$)** e **Value ($W^V$)** captura a essência das mudanças contextuais necessárias para uma nova tarefa.

Como a nossa implementação do `MultiHeadAttention` expõe as projeções de forma limpa através de atributos (`self.proj_q` e `self.proj_v`), podemos percorrer os blocos do *Encoder* e substituir as camadas `layers.Dense` originais pela nossa `LoRADense`, passando a camada original como parâmetro.

In [ ]:
# vamos iterar pelas camadas de Encoder do MiniBERT:

rank_lora = 4
alpha_lora = 8

print("Iniciando a injeção cirúrgica do LoRA nas camadas de Atenção...")

for i, encoder_layer in enumerate(base_minibert.encoder_layers):
    mha_block = encoder_layer.mha

    # pegando a dimensão de entrada para construir o LoRA corretamente:
    input_dim = config["d_model"]

    # para reescrever uma camada no Keras, precismos "destrancar o rastreado":
    mha_block._tracker.locked = False

    # instanciando as nossas camadas LoRADense encapsulando as originais:
    lora_q = LoRADense(mha_block.proj_q, r=rank_lora, alpha=alpha_lora)
    lora_v = LoRADense(mha_block.proj_v, r=rank_lora, alpha=alpha_lora)

    # instanciando os pesos A e B de forma explícita:
    lora_q.build((None, input_dim))
    lora_v.build((None, input_dim))

    # realizando a substituição:
    mha_block.proj_q = lora_q
    mha_block.proj_v = lora_v

    # por fim, "trancamos o rastreador" para manter a segurança dessa camada:
    mha_block._tracker.locked = True

    print(f" -> LoRA injetado com sucesso no Encoder Bloco {i} (Query e Value).")

print("\nProcesso concluído!")

### **3.3 Conectando o Head de Classificação para o IMDB:**
<font size=3>

O modelo BERT foi desenhado para extrair a representação semântica agregada de uma frase inteira através do seu primeiro token especial, o **`[CLS]`**.

Na aula passada, estruturamos o `MiniBERT` para expor o `pooled_output`, que captura o vetor do token `[CLS]` após passar pelas $N$ camadas do encoder e aplica uma transformação linear com ativação de tangente hiperbólica ($\tanh$). Para realizar a classificação binária de sentimentos (*crítica positiva* vs. *crítica negativa*), precisamos apenas conectar uma camada linear final com ativação **Sigmoid** sobre este `pooled_output`.


In [ ]:
# definindo as três entradas do modelo:
input_ids = layers.Input(shape=(config['max_len'],), dtype=tf.int32, name="input_ids")
segment_ids = layers.Input(shape=(config['max_len'],), dtype=tf.int32, name="segment_ids")
mask = layers.Input(shape=(config['max_len'],), dtype=tf.int32, name="mask") # <-- Faltava a máscara!

# passando as entradas pelo nosso MiniBERT:
_, pooled_output = base_minibert((input_ids, segment_ids, mask))

# adicionando uma camada de Dropout para regularização:
x = layers.Dropout(config['dropout_rate'])(pooled_output)

# head de classificação binária:
x_out = layers.Dense(1, activation='sigmoid', name="sentiment_head")(x)

# instanciando o modelo final:
imdb_lora_model = Model(inputs=[input_ids, segment_ids, mask], outputs=x_out, name="MiniBERT_LoRA_IMDB")

imdb_lora_model.summary()

## **4. O _pipeline_ de dados e treinamento:**
<font size=3>
    
Nesta seção, consolidaremos o nosso pipeline de ponta a ponta. Iremos carregar o tradicional dataset IMDB para análise de sentimento, preparar os dados de texto estruturando as três entradas exigidas pelo `MiniBERT` (`input_ids`, `segment_ids` e `attention_mask`) e, finalmente, treinar os nossos adaptadores LoRA.

### **4.1 Pré-processamento dos dados:**
<font size=3>

Para alimentar o nosso modelo, precisamos de garantir que os textos das críticas sejam processados exatamente com o mesmo tokenizador WordPiece utilizado na fase de pré-treinamento. Ajustaremos o comprimento máximo das sequências para `max_len = 64`.


In [ ]:
import pandas as pd
from transformers import BertTokenizerFast
from datasets import Dataset

In [ ]:
# importando os dados do IMDB:
df = pd.read_csv("dataset/imdb.csv")

# Extração para listas
X = df["reviews"].tolist()
y = df["label"].tolist()

# dividindo os dados:
N_samples = len(X)
N_train = int(0.7 * N_samples)
N_val = int(0.2 * N_samples)

X_train = X[:N_train]
y_train = y[:N_train]
print("X-train:", len(X_train))

X_val = X[N_train:N_train+N_val]
y_val = y[N_train:N_train+N_val]
print("X-val:", len(X_val))

X_test = X[N_train+N_val:]
y_test = y[N_train+N_val:]
print("X-test:", len(X_test))

In [ ]:
del df, X, y

In [ ]:
# convertendo as listas para objetos 'Dataset' da Hugging Face:
train_raw = Dataset.from_dict({"text": X_train, "label": y_train})
val_raw = Dataset.from_dict({"text": X_val, "label": y_val})
test_raw = Dataset.from_dict({"text": X_test, "label": y_test})

# definindo o tokenizador:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
config['vocab_size'] = tokenizer.vocab_size
batch_size = 64

print("vocab-size:", config['vocab_size'])

def tokenize_fun(samples):
    return tokenizer(samples["text"], padding="max_length", truncation=True,
        max_length=config['max_len'], return_special_tokens_mask=True)

# aplicando a tokenização:
train_tokenized = train_raw.map(tokenize_fun, batched=True, remove_columns=["text"])
val_tokenized = val_raw.map(tokenize_fun, batched=True, remove_columns=["text"])
test_tokenized = test_raw.map(tokenize_fun, batched=True, remove_columns=["text"])

In [ ]:
del X_train, y_train
del X_val, y_val
del X_test, y_test

In [ ]:
# exportando para TF Dataset:
train_tf_raw = train_tokenized.to_tf_dataset(columns=["input_ids", "token_type_ids", "attention_mask", "special_tokens_mask"],
                                             label_cols=["label"], shuffle=True, batch_size=batch_size)



val_tf_raw = val_tokenized.to_tf_dataset(columns=["input_ids", "token_type_ids", "attention_mask", "special_tokens_mask"],
                                          label_cols=["label"], shuffle=True, batch_size=batch_size)

test_tf_raw = test_tokenized.to_tf_dataset(columns=["input_ids", "token_type_ids", "attention_mask", "special_tokens_mask"],
                                           label_cols=["label"], shuffle=True, batch_size=batch_size)


In [ ]:
del train_raw, val_raw, test_raw
del train_tokenized, val_tokenized, test_tokenized

In [ ]:
# Ajustando de assinatura para o MiniBert:
# O Hugging Face gera a chave "token_type_ids", mas nosso modelo espera "segment_ids".

def format_inputs(features, labels):
    inputs = {"input_ids": features["input_ids"],
              "segment_ids": features["token_type_ids"],
              "mask": features["attention_mask"]}

    return inputs, labels

train_tf = train_tf_raw.map(format_inputs)
val_tf = val_tf_raw.map(format_inputs)
val_tf = test_tf_raw.map(format_inputs)

In [ ]:
del train_tf_raw, val_tf_raw, test_tf_raw

### **4.2 Compilação do modelo:**
<font size=3>

Com os nossos dados preparados, vamos compilar o modelo `imdb_lora_model`. Utilizaremos o otimizador **Adam** e a função de perda **BinaryCrossentropy** (apropriada para classificação binária de sentimentos).

Nesta etapa, configuraremos o otimizador com um *learning rate* bem baixo. Valores como $10^{-3}$ ou superiores são comuns para treinar modelos do zero. No entanto, no cenário de **_fine-tuning_**, adotar uma taxa de aprendizado contida é uma decisão arquitetural crítica pelas seguintes razões:
1. **Evitar o esquecimento catastrófico:** O nosso `MiniBERT` passou por um pré-treinamento pesado via MLM, onde aprendeu as estruturas complexas, a semântica e a sintaxe do idioma. Se utilizássemos um *learning rate* muito alto, os passos do otimizador seriam agressivos demais. Isso destruiria as representações internas já consolidadas na tentativa de se adequar rapidamente ao dataset IMDB, fazendo o modelo esquecer o conhecimento linguístico prévio.

2. **O "ponto ideal" para adaptadores PEFT:** No LoRA, todo o corpo do BERT está congelado, e estamos atualizando apenas a nova cabeça de classificação e as pequenas matrizes $A$ e $B$. Como a matriz $B$ é inicializada com valores **zerados**, a atualização inicial ($\Delta W = B \cdot A$) começa em zero. Uma taxa de $2\times 10^{-4}$ funciona como um excelente intermediário: é baixa o suficiente para manter a estabilidade perto da solução ideal que o BERT já ocupa, mas ligeiramente maior do que as taxas de um *Full Fine-Tuning* tradicional (que costumam usar $2\times 10^{-5}$ a $5\times 10^{-5}$), dando o fôlego necessário para que os novos adaptadores saiam do zero e convirjam rapidamente.
   

In [ ]:
import numpy as np
from keras.optimizers import Adam

In [ ]:
# realizando 'dummy forward pass' para inicializar variáveis do LoRA:
for dummy_batch in train_tf.take(1):
    dummy_x, dummy_y = dummy_batch
    break

_ = imdb_lora_model(dummy_x) # variáveis inicializadas!

imdb_lora_model.compile(optimizer=Adam(learning_rate=2e-4), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# 2. Auditoria e Contagem de Parâmetros Treináveis na unha:
total_params = sum([np.prod(p.shape) for p in imdb_lora_model.get_weights()])
trainable_params = sum([np.prod(p.shape) for p in imdb_lora_model.trainable_weights])
non_trainable_params = sum([np.prod(p.shape) for p in imdb_lora_model.non_trainable_weights])

print(f"Parâmetros totais do modelo: {total_params:,}")
print(f"Parâmetros treináveis (LoRA + Head): {trainable_params:,}")
print(f"Parâmetros congelados (Base BERT):   {non_trainable_params:,}")
print(f"Percentagem de parâmetros treináveis: {(trainable_params/total_params)*100:.3f}%")

### **4.3 Executando o _fine-tuning_:**
<font size=3>

Agora, vamos executar o método *fine-tuning*. Como o corpo do nosso BERT está completamente congelado e apenas as pequenas matrizes $A$ e $B$ do LoRA (juntamente com o cabeçalho de classificação) serão atualizadas, o tempo de processamento por época será drasticamente reduzido, exigindo muito menos memória da GPU.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
report = imdb_lora_model.fit(train_tf, epochs=3, validation_data=val_tf, verbose=1)

In [ ]:

ax1.plot(report.history['loss'], label='Treino (LoRA)', color='tab:blue', linewidth=2, marker='o')
ax1.plot(report.history['val_loss'], label='Validação', color='tab:orange', linewidth=2, linestyle='--', marker='s')
ax1.set_title('Convergência da Função de Perda (Loss)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Épocas', fontsize=10)
ax1.set_ylabel('Loss', fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(fontsize=10)

ax2.plot(report.history['accuracy'], label='Treino (LoRA)', color='tab:green', linewidth=2, marker='o')
ax2.plot(report.history['val_accuracy'], label='Validação', color='tab:red', linewidth=2, linestyle='--', marker='s')
ax2.set_title('Evolução da Acurácia', fontsize=12, fontweight='bold')
ax2.set_xlabel('Épocas', fontsize=10)
ax2.set_ylabel('Acurácia', fontsize=10)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## **Considerações Finais**
<font size=3>

Neste notebook, construímos uma ponte robusta entre a teoria matemática profunda e a prática da engenharia de *Deep Learning*. Partimos de um modelo fundacional pré-treinado do zero (o nosso `MiniBERT`) e, em vez de adotarmos o caminho tradicional e custoso do *Full Fine-Tuning*, realizamos a injetação de adaptadores **LoRA** (*Low-Rank Adaptation*).

1. **Eficiência extrema (PEFT):** Comprovamos na prática — através de auditorias de parâmetros e gráficos de convergência — como atualizar menos de 1% da rede é suficiente para adaptar um modelo a uma nova tarefa complexa (como a análise de sentimentos do IMDB), economizando drasticamente memória de GPU e tempo computacional.

2. **Combate ao esquecimento catastrófico:** Ao congelar todo o "corpo" do BERT, garantimos que a sintaxe e a semântica aprendidas a duras penas durante a fase de *Masked Language Modeling* fossem totalmente preservadas.

3. **Domínio de infraestrutura:** Fomos além do uso de bibliotecas de alto nível. Manipulamos o estado interno do grafo do Keras, construímos camadas customizadas (`LoRADense`) baseadas em fatoração de matrizes e integramos o ecossistem da Hugging Face com pipelines de alta performance via `tf.data`.

Dominar o paradigma **PEFT** não é apenas um exercício acadêmico; é uma competência altamente requisitada no mercado atual. A mesma matemática de baixo posto ($B \cdot A$) que vocês codificaram forma a espinha dorsal de como a indústria adapta gigantescos Modelos de Linguagem (como Llama 3, Mistral e Gemma) para casos de uso corporativos utilizando hardware de consumo acessível.
